# 05 — Inventory Optimization

**Milestone 12** — converting the validated MA-28 demand forecast into
inventory decisions: safety stock, reorder points and a lead-time-respecting
backtesting simulation.

⚠ **Data vs assumptions:** M5 provides daily sales only. All lead times,
service-level targets, starting inventories and order quantities below are
**scenario assumptions for inventory simulation**, not observed data.

## 1. Business Objective

Answer: *given expected demand and forecast uncertainty, how much inventory
should the company hold and when should it reorder?*

Approach: MA-28 forecast (validation winner from the Milestone-11 audit) →
forecast-error statistics (train period only) → statistical safety stock →
reorder points → (s, Q) backtesting simulation across a 3×3 grid of
service-level × lead-time scenarios → ABC prioritization → recommendations.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from inventory.inventory_optimizer import (
    LEAD_TIME_SCENARIOS, SERVICE_LEVEL_SCENARIOS, Z_SCORES, RANDOM_STATE,
    calculate_forecast_error, calculate_safety_stock, calculate_reorder_point,
    classify_abc, simulate_inventory, calculate_inventory_metrics,
    load_demand_data, compute_series_error_stats, build_inventory_policy,
    run_scenario_analysis, select_business_case_series, PROCESSED,
)

np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 160)
print("config:", LEAD_TIME_SCENARIOS, SERVICE_LEVEL_SCENARIOS, Z_SCORES)

## 2. Forecast Input

Leakage-safe rolling MA-28: `forecast(t) = mean(demand[t-28:t-1])` — only
demand strictly before `t`. This is the exact logic that won the audited
validation comparison; the invalid frozen MA-28 from the earlier ML script
is deliberately NOT used.

In [ ]:
df = load_demand_data()
print(f"{len(df):,} rows, {df['id'].nunique()} series, "
      f"{df['date'].min().date()} -> {df['date'].max().date()}")
print(df.groupby("split").agg(rows=("demand", "size"),
      start=("date", "min"), end=("date", "max")))

### 2.1 Demand vs forecast — example series

In [ ]:
example_id = "FOODS_3_090_CA_3"  # high-demand example series
if example_id not in set(df["id"]):
    example_id = df.groupby("id")["demand"].sum().idxmax()
g = df[(df["id"] == example_id) & (df["date"] >= "2015-10-19")]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(g["date"], g["demand"], label="actual demand", lw=1)
ax.plot(g["date"], g["ma28_forecast"], label="MA-28 forecast", lw=1.2)
ax.set_title(f"Demand vs MA-28 forecast — {example_id} (test window)")
ax.set_ylabel("units/day"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_demand_vs_forecast.png", dpi=110)
plt.close(fig)
print("saved fig_demand_vs_forecast.png")

## 3. Inventory Assumptions

| Item | Value | Status |
|---|---|---|
| Lead times | 3 / 7 / 14 days | **scenario assumption** |
| Service levels | 90% / 95% / 99% | **scenario assumption** |
| Starting inventory | reorder point on day 0 | **simulation assumption** |
| Order quantity | max(1, ceil(forecast × lead_time)) | **simulation assumption** |
| Unmet demand | lost sales (not backordered) | **simulation assumption** |

σ (safety-stock variability input) = std of forecast error on TRAIN data
only (≤ 2015-04-13). No test-period demand informs any policy parameter.

In [ ]:
error_stats = compute_series_error_stats(df)
print(error_stats[["n", "MAE", "RMSE", "mean_error", "std_error"]]
      .describe().loc[["mean", "min", "max"]])
print("\nlast error date used:", error_stats["last_error_date"].max().date(),
      "(train period only)")

## 4. Forecast Error

`forecast_error = actual - forecast`, summarized per series. Distribution
of per-series σ — the direct driver of safety stock:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(error_stats["std_error"], bins=30, color="#4c78a8", edgecolor="white")
ax.set_title("Per-series forecast-error std (train period)")
ax.set_xlabel("sigma"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_sigma_hist.png", dpi=110)
plt.close(fig)
print(f"sigma: mean={error_stats['std_error'].mean():.2f}, "
      f"median={error_stats['std_error'].median():.2f}, "
      f"max={error_stats['std_error'].max():.2f}")

## 5. Safety Stock

`SafetyStock = Z × σ × sqrt(lead_time)` with Z = 1.282 / 1.645 / 2.326 for
90% / 95% / 99%. Safety stock rises with service level, with σ, and with
the square root of lead time.

In [ ]:
sls = [0.90, 0.95, 0.99]
lts = [3, 7, 14]
sig = error_stats["std_error"].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for sl in sls:
    axes[0].plot(lts, [calculate_safety_stock(Z_SCORES[sl], sig, lt)
                       for lt in lts], "o-", label=f"{int(sl*100)}% SL")
axes[0].set_title(f"Safety stock vs lead time (mean sigma={sig:.2f})")
axes[0].set_xlabel("lead time (days)"); axes[0].set_ylabel("units")
axes[0].legend(); axes[0].grid(alpha=0.3)
sigmas = np.linspace(0, 5, 50)
for sl in sls:
    axes[1].plot(sigmas, [calculate_safety_stock(Z_SCORES[sl], s, 7)
                          for s in sigmas], label=f"{int(sl*100)}% SL")
axes[1].set_title("Safety stock vs variability (7-day lead time)")
axes[1].set_xlabel("sigma"); axes[1].set_ylabel("units")
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_safety_stock.png", dpi=110)
plt.close(fig)
print("saved fig_safety_stock.png")

## 6. Reorder Point

`ROP = forecast × lead_time + safety stock` (each term rounded up). ROP
grows linearly with lead time through lead-time demand and sub-linearly
through safety stock.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
forecasts = np.linspace(0.5, 8, 40)
for lt in lts:
    ax.plot(forecasts, [calculate_reorder_point(f, lt,
            calculate_safety_stock(Z_SCORES[0.95], sig, lt))
            for f in forecasts], label=f"ROP, LT={lt}d")
ax.set_title("Reorder point vs forecast, by lead time (95% SL)")
ax.set_xlabel("daily forecast (units)"); ax.set_ylabel("ROP (units)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_reorder_point.png", dpi=110)
plt.close(fig)
print("saved fig_reorder_point.png")

## 7. Inventory Policy

For every (id, date) and every scenario we compute the full policy:
`forecast`, `lead_time`, `service_level`, `lead_time_demand`,
`safety_stock`, `reorder_point`. Saved to
`data/processed/inventory_policy.parquet`.

In [ ]:
policy = pd.read_parquet("../data/processed/inventory_policy.parquet")
print(f"{len(policy):,} policy rows "
      f"(113,100 id-dates x 9 scenarios)")
policy.sample(5, random_state=RANDOM_STATE)

## 8. Backtesting Simulation

For each series and scenario we simulate day by day: receive arrivals →
demand occurs (unmet demand lost) → reorder if inventory position ≤ ROP →
order arrives after exactly `lead_time` days. Starting inventory = ROP on
day 0 (simulation assumption). The scenario grid comes from
`run_scenario_analysis`; below we re-run a single-series trace.

In [ ]:
scen = pd.read_csv("../data/processed/inventory_scenarios.csv")
per_series = pd.read_parquet("../data/processed/inventory_per_series.parquet")

# single-series trace: 95% SL, 7-day lead time, validation window
sid = example_id
gdf = df[(df["id"] == sid) & (df["split"] == "validation")]
pol = policy[(policy["id"] == sid) & (policy["service_level"] == 0.95)
             & (policy["lead_time"] == 7)].sort_values("date")
sim = simulate_inventory(
    demand=gdf["demand"].to_numpy(float),
    forecast=gdf["ma28_forecast"].to_numpy(float),
    reorder_point=pol["reorder_point"].to_numpy(int),
    lead_time_days=7,
)
print(calculate_inventory_metrics(sim))
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(gdf["date"].to_numpy(), sim["ending_inventory"],
        label="ending inventory", color="#4c78a8")
ax.plot(gdf["date"].to_numpy(), pol["reorder_point"],
        label="reorder point", color="#e45756", ls="--", lw=1)
so = sim["stockout_units"] > 0
ax.scatter(gdf["date"].to_numpy()[so], sim.loc[so, "ending_inventory"],
           color="black", zorder=5, s=18, label="stockout day")
ax.set_title(f"Simulated inventory — {sid} (95% SL, 7-day LT)")
ax.set_ylabel("units"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_simulation_trace.png", dpi=110)
plt.close(fig)
print("saved fig_simulation_trace.png")

## 9. Service-Level Analysis

Actual fill rate achieved in the simulation vs the target service level.
Higher targets deliver higher realized fill rates at the cost of more
inventory (validated below, not assumed).

In [ ]:
val = scen[scen["window"] == "validation"]
print(val.pivot(index="lead_time", columns="service_level",
      values="service_level_actual_weighted").round(4))

## 10. Lead-Time Analysis

Longer lead times raise both the lead-time demand and the safety-stock
buffer, hence a higher reorder point and more inventory held.

In [ ]:
print("Average inventory (units, sum over series):")
print(val.pivot(index="lead_time", columns="service_level",
      values="average_inventory").round(0))
print("\nStockout rate (days with unmet demand / all days):")
print(val.pivot(index="lead_time", columns="service_level",
      values="stockout_rate").round(4))

### 10.1 Service level vs average inventory (the inventory/service trade-off)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lt in lts:
    sub = val[val["lead_time"] == lt].sort_values("service_level")
    ax.plot(sub["service_level"] * 100, sub["average_inventory"], "o-",
            label=f"LT={lt}d")
ax.set_title("Service level vs average inventory (validation)")
ax.set_xlabel("target service level (%)")
ax.set_ylabel("average inventory (units, all series)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_service_vs_inventory.png", dpi=110)
plt.close(fig)
print("saved fig_service_vs_inventory.png")

## 11. ABC Analysis

Analytical classification by cumulative demand contribution on TRAIN
demand: A = first 80%, B = next 15%, C = last 5% of demand. This connects
the forecasting work to inventory prioritization.

In [ ]:
abc = pd.read_csv("../data/processed/inventory_abc_series.csv")
abc_perf = pd.read_csv("../data/processed/inventory_abc_performance.csv")
summary = abc.groupby("abc_class").agg(
    series=("id", "size"),
    demand_share=("demand_percentage", "sum"),
    avg_daily_demand=("total_demand", lambda s: s.mean()),
)
print(summary)

abc_sorted = abc.sort_values("cumulative_demand_percentage", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(abc_sorted) + 1),
        abc_sorted["cumulative_demand_percentage"] * 100, color="#4c78a8")
for cut, lbl in [(0.80, "A: 80% of demand"), (0.95, "B: next 15%")]:
    ax.axhline(cut * 100, color="#e45756", ls="--", lw=1)
ax.set_title("ABC demand contribution (cumulative % vs series ranked)")
ax.set_xlabel("series rank (high -> low demand)")
ax.set_ylabel("cumulative demand (%)"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("../data/processed/fig_abc_contribution.png", dpi=110)
plt.close(fig)
print("saved fig_abc_contribution.png")

vp = abc_perf[abc_perf["window"] == "validation"]
print("\nInventory performance by ABC class (validation, 95% SL / 7d LT):")
print(vp[(vp["service_level"] == 0.95) & (vp["lead_time"] == 7)]
      [["abc_class", "total_demand", "stockout_rate",
        "average_inventory", "inventory_turnover"]])

## 12. Inventory Trade-offs

Expected relationships, validated against the simulation:

* higher service level → higher safety stock → higher average inventory →
  lower stockout risk
* longer lead time → higher lead-time demand → higher reorder point

In [ ]:
trade = val.copy()
trade["safety_stock_mean"] = trade["safety_stock_mean"].round(2)
cols = ["service_level", "lead_time", "safety_stock_mean",
        "average_inventory", "stockout_rate"]
print(trade[cols].sort_values(["lead_time", "service_level"])
      .to_string(index=False))

mon = trade.sort_values(["lead_time", "service_level"]).reset_index(drop=True)
assert (mon.groupby("lead_time")["average_inventory"]
        .apply(lambda s: s.is_monotonic_increasing)).all(), \
    "higher SL must not reduce average inventory"
assert (mon.groupby("service_level")["safety_stock_mean"]
        .apply(lambda s: s.is_monotonic_increasing)).all(), \
    "longer LT must not reduce safety stock"
print("\ntrade-off relationships confirmed by simulation")

## 13. Business Recommendations

Recommended scenario: on the validation Pareto frontier of (weighted fill
rate ↑, average inventory ↓), the cheapest scenario reaching ≥ 98% fill.
Evidence-based, per ABC class:

In [ ]:
bc = pd.read_csv("../data/processed/inventory_business_cases.csv")
print("Business-case series (recommended scenario):")
print(bc.round(3).to_string(index=False))
print()
recs = pd.read_csv("../data/processed/inventory_recommendations.csv")
print(recs.to_string(index=False))

### 13.1 How the three business cases differ

The same service-level target translates into very different buffers: the
high-variability volatile series needs the largest safety stock and ROP;
the stable series needs the smallest relative buffer; the intermittent
series holds little stock but is exposed to lumpy demand.

In [ ]:
per_case = bc[bc["window"] == "validation"]
print(per_case[["case", "mean_demand", "std_demand", "mean_forecast",
      "mean_safety_stock", "mean_reorder_point", "average_inventory",
      "stockout_rate"]].to_string(index=False))

## 14. Limitations

* M5 provides **no inventory-on-hand data** — starting inventory is an
  assumption (ROP on day 0), not observed.
* Lead times (3/7/14 days) and service-level targets (90/95/99%) are
  **scenario assumptions**, not Walmart/M5 data.
* Order quantity Q and the lost-sales (no backorder) assumption are
  simulation choices.
* No cost data: the trade-off is shown in units, not money; no cost
  optimization was performed.
* σ comes from MA-28 training errors only; if demand patterns shift, the
  buffers are mis-sized (no adaptive re-estimation).
* Results are policy simulation on a 300-series FOODS development subset,
  not historical company inventory performance.
* The simulated inventory turnover is not comparable to real retail
  turnover.

## 15. Conclusion

* The MA-28 forecast converts cleanly into an operational (s, Q) policy:
  safety stock and reorder points scale with variability and lead time as
  the standard formulas predict, confirmed in simulation.
* Validation-based recommendation: on the fill-rate / inventory Pareto
  frontier, the cheapest scenario reaching ≥ 98% weighted fill rate is
  the recommended default policy; 90% / 3-day holds the least inventory
  with the highest stockout exposure; 99% / 7-day maximizes fill rate at
  the highest inventory cost.
* A-items (~80% of demand on ~32% of series) justify tight monitoring and
  higher service targets; C-items (mostly intermittent, near-zero demand)
  should not carry heavy buffers — the Milestone-11 audit and the ABC
  analysis agree.
* All conclusions are simulation-based under documented assumptions and
  must be re-validated with real lead times, costs and inventory data
  before operational use.